In [1]:
from ultralytics import YOLO
import cv2
from ultralytics.trackers.byte_tracker import BYTETracker
from types import SimpleNamespace
import numpy as np

# tracker
args = SimpleNamespace(
    track_high_thresh=0.5,
    track_low_thresh=0.1,
    new_track_thresh=0.6,
    track_buffer=30,
    match_thresh=0.8,
    fuse_score=True,
    mot20=False
)

tracker = BYTETracker(args, frame_rate=30)

# load the model
model = YOLO('yolo11s.pt')

video_path = '2.webm'

cap = cv2.VideoCapture(video_path)

# find static positions
lk_params = dict(winSize=(15, 15),maxLevel = 2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

prev_frame_gray = None
prev_points = None

# read video
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # understand camera movement (lite version of SLAM)
    camera_dx, camera_dy = 0, 0

    if prev_frame_gray is not None:
        if prev_points is None or len(prev_points) < 10:
            prev_points = cv2.goodFeaturesToTrack(prev_frame_gray, maxCorners=100,
                                                 qualityLevel=0.3, minDistance=7) # find landmarks

        # calculate where these points in a new frame
        if prev_points is not None:
            new_points, status, error = cv2.calcOpticalFlowPyrLK(prev_frame_gray, frame_gray,
                                                            prev_points, None, **lk_params)

            if new_points is not None and len(new_points) > 0:
                # calculate average offset
                good_new = new_points[status == 1]
                good_old = prev_points[status == 1]
                if len(good_new) > 0:
                    diffs = good_new - good_old
                    camera_dx, camera_dy = np.mean(diffs, axis=0) # the main idea - calculate average offset
                    prev_points = good_new.reshape(-1, 1, 2)
                else:
                    prev_points = None
            else:
                prev_points = None

    results = model(frame, device='cpu', conf=0.35, classes=[0], verbose=False)[0]

    # update tracker
    tracks = tracker.update(results.boxes, frame)

    for track in tracks:
        x1, y1, x2, y2, object_id = track[:5]

        # get concrete track by id
        target_track = next((t for t in tracker.tracked_stracks if t.track_id == object_id), None)

        if target_track is not None and target_track.state == 1: # 1 — Tracked status
            mean = target_track.mean
            vx = mean[4] # speed by x
            vy = mean[5] # speed by y

            # true speed of people - (speed of in frame - speed of camera)
            true_vx = vx - camera_dx
            true_vy = vy - camera_dy

            # center of frame
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2

            # prediction of future position
            future_x = cx + true_vx * 30
            future_y = cy + true_vy * 30

            # plot line with predictions
            cv2.line(frame, (int(cx), int(cy)), (int(future_x), int(future_y)), (0, 0, 255), 3)
            cv2.circle(frame, (int(future_x), int(future_y)), 5, (0, 0, 255), -1)

        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(frame, f"ID: {int(object_id)}", (int(x1), int(y1) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.putText(frame, f"Cam Shift: x={camera_dx:.1f} y={camera_dy:.1f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.imshow("Motion Predictor", frame)

    prev_frame_gray = frame_gray.copy()

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()